# 音乐源分离 - pymss WebUI / ComfyUI

This notebook installs [pymss](https://github.com/pymss-project/pymss) (Python Music Source Separation) on Google Colab with GPU, and supports both **pymss WebUI** and **[ComfyUI](https://github.com/comfyanonymous/ComfyUI)** + [comfy-mss](https://github.com/pymss-project/comfy-mss) nodes.

**What you get:** A browser-accessible interface where you can upload audio and separate vocals, drums, bass, and other instruments.

**Persistence:** Step 2 mounts Google Drive and points models, output, input, ComfyUI settings and the HuggingFace cache at `/content/drive/MyDrive/pymss`. A Colab runtime is wiped on disconnect, so this is what keeps your generated tracks after the session ends — separated audio is written there and downloaded directly from Google Drive.

**Requirements:** Colab runtime with GPU (Runtime > Change runtime type > T4 GPU).

> ⚠️ **Known Issue:** WebUI file upload/download may not work correctly through Cloudflare tunnel. This is likely a pymss WebUI frontend issue, not caused by this notebook. Use REST API or Python API as alternative.

---
*Author: [uwuAitry](https://github.com/uwuAitry) | Based on [pymss](https://github.com/pymss-project/pymss) by pymss-project*

## 1. Install dependencies

In [ ]:
#@title 1. Install dependencies { display-mode: "form" }
# ============================================================
# pymss Colab - Step 1: Install
# ============================================================

import os, subprocess, shutil, sys
from importlib.metadata import version as _pkg_version, PackageNotFoundError

# ---- PyTorch floor required by pymss (torch>=2.7.1) ----
# Checked via installed metadata, BEFORE `import torch`, so that a version bump
# cannot swap the module underneath an already-imported torch.
def _ver_tuple(s):
    return tuple(int(p) if p.isdigit() else 0 for p in s.split(".")[:3])

TORCH_MIN = "2.7.1"
try:
    torch_installed = _pkg_version("torch")
except PackageNotFoundError:
    torch_installed = None

if torch_installed is None or _ver_tuple(torch_installed.split("+")[0]) < _ver_tuple(TORCH_MIN):
    print(f"[torch] {torch_installed or 'not installed'} < {TORCH_MIN} (pymss requirement) -> installing CUDA build")
    !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu128
    import importlib
    importlib.invalidate_caches()
    try:
        torch_installed = _pkg_version("torch")
    except PackageNotFoundError:
        pass
else:
    print(f"[torch] {torch_installed} (>= {TORCH_MIN}, OK)")

import torch  # imported only after the version check above

# ---- Check GPU ----
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime > Change runtime type > GPU.")
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"[GPU] {gpu_name} ({gpu_mem:.1f} GB)")

# ---- Install pymss[server] ----
!pip install -q "pymss[server]"
pymss_ver = subprocess.check_output(["pip", "show", "pymss"], text=True)
pymss_ver = [l.split(":", 1)[1].strip() for l in pymss_ver.splitlines() if l.startswith("Version:")][0]
print(f"[pymss] {pymss_ver}")

# ---- Download cloudflared ----
if not shutil.which("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
cf_ver = subprocess.check_output(["cloudflared", "--version"], text=True).strip()
print(f"[cloudflared] {cf_ver}")

# ---- Install ComfyUI + comfy-mss (optional, for comfyui mode) ----
COMFYUI_DIR = "/content/ComfyUI"
if not os.path.isdir(COMFYUI_DIR):
    !git clone https://github.com/comfyanonymous/ComfyUI.git {COMFYUI_DIR}
!pip install -q -r {COMFYUI_DIR}/requirements.txt
# NOTE: comfyui-frontend-package is pinned inside ComfyUI/requirements.txt.
# Do NOT install it separately with an unpinned version -- that shadows the
# upstream pin and causes frontend / tunnel asset drift.
# comfy-mss custom node
comfy_mss_dir = os.path.join(COMFYUI_DIR, "custom_nodes", "comfy-mss")
if not os.path.isdir(comfy_mss_dir):
    !git clone https://github.com/pymss-project/comfy-mss.git {comfy_mss_dir}
# ComfyUI Manager, plus its own runtime deps (GitPython / PyGithub / matrix-nio / nh3 / uv ...)
manager_dir = os.path.join(COMFYUI_DIR, "custom_nodes", "ComfyUI-Manager")
if not os.path.isdir(manager_dir):
    !git clone https://github.com/Comfy-Org/ComfyUI-Manager.git {manager_dir}
!pip install -q -r {manager_dir}/requirements.txt
# comfyui-output-swap: frontend-only UX pack (drag one output onto another
# output slot to re-home all its downstream links). web/dist/ is committed
# upstream, so no `bun run build` is needed; needs frontend >= 1.43 for the
# canvas.linkConnector API and no-ops safely if it is missing.
swap_dir = os.path.join(COMFYUI_DIR, "custom_nodes", "comfyui-output-swap")
if not os.path.isdir(swap_dir):
    !git clone https://github.com/laurigates/comfyui-output-swap.git {swap_dir}
# NOTE: ComfyUI Manager needs security_level = weak to accept --listen 0.0.0.0.
# That config.ini lives under ComfyUI's *user directory*, which Step 2 may move
# onto Google Drive -- so Step 3 writes it, once the final path is known.

# ---- UniverSR + comfy-universr node (audio super-resolution) ----
# NOT source separation: UniverSR extends full-mix bandwidth to 48 kHz.
# custom_nodes/ is not tracked upstream, so the node is written inline here
# rather than cloned -- Colab only needs this notebook.
universr_dir = os.path.join(COMFYUI_DIR, "custom_nodes", "comfy-universr")
os.makedirs(universr_dir, exist_ok=True)
if subprocess.run([sys.executable, "-c", "import universr"], capture_output=True).returncode != 0:
    !pip install -q "git+https://github.com/woongzip1/UniverSR.git"
# r-string: the node source contains \n escapes that must stay literal.
_COMFY_UNIVERSR_SRC = r'''"""ComfyUI custom node for UniverSR (audio super-resolution).

UniverSR is NOT a music source separation model. It is an audio
super-resolution / bandwidth-extension model: it takes full-mix audio and
upscales 8/12/16/24 kHz content to 48 kHz via vocoder-free flow matching.
For source separation use the comfy-mss nodes instead.

Official repo: https://github.com/woongzip1/UniverSR (MIT License)
"""

import os
import tempfile

import torch

INPUT_SRS = ["8000", "12000", "16000", "24000"]
ODE_METHODS = ["euler", "midpoint", "rk4"]
MODEL_CHOICES = ["woongzip1/universr-audio", "woongzip1/universr-speech"]
OUTPUT_SR = 48000  # UniverSR always outputs 48 kHz

# Process-lifetime model cache: (repo_id, device) -> UniverSR instance.
_MODELS = {}


def _resolve_device(device):
    device = (device or "").strip()
    if device and device.lower() != "auto":
        return device
    return "cuda" if torch.cuda.is_available() else "cpu"


def _get_model(repo_id, device):
    key = (repo_id, device)
    if key not in _MODELS:
        try:
            # Lazy import: ComfyUI can still start when the package is missing.
            from universr import UniverSR
        except ImportError as exc:
            raise RuntimeError(
                "The 'universr' package is not installed. Install it with:\n"
                "  pip install git+https://github.com/woongzip1/UniverSR.git\n"
                "(autodl_setup.sh in the pymss-for-colab repo does this for you)"
            ) from exc
        print(f"[UniverSR Enhance] Loading {repo_id} on {device} ...")
        _MODELS[key] = UniverSR.from_pretrained(repo_id, device=device)
    return _MODELS[key]


class UniverSREnhance:
    """Enhance full-mix AUDIO (8/12/16/24 kHz bandwidth) to 48 kHz with UniverSR."""

    @classmethod
    def INPUT_TYPES(cls):
        return {
            "required": {
                "audio": ("AUDIO",),
                "model": (MODEL_CHOICES,),
                "input_sr": (INPUT_SRS,),
                "ode_method": (ODE_METHODS, {"default": "midpoint"}),
                "ode_steps": ("INT", {"default": 4, "min": 1, "max": 64, "step": 1}),
                "guidance_scale": ("FLOAT", {"default": 1.5, "min": 0.0, "max": 10.0, "step": 0.05}),
            },
            "optional": {
                "device": ("STRING", {"default": "auto", "tooltip": "torch device: auto / cuda / cuda:0 / cpu"}),
            },
        }

    RETURN_TYPES = ("AUDIO",)
    RETURN_NAMES = ("audio",)
    FUNCTION = "enhance"
    CATEGORY = "audio"
    DESCRIPTION = (
        "UniverSR audio super-resolution: extends full-mix audio bandwidth "
        "(declared via input_sr: 8/12/16/24 kHz) to 48 kHz via flow matching. "
        "NOT source separation - use comfy-mss nodes for that. "
        "guidance_scale 0 disables classifier-free guidance."
    )

    def enhance(self, audio, model, input_sr, ode_method, ode_steps, guidance_scale, device="auto"):
        try:
            import torchaudio
        except ImportError as exc:
            raise RuntimeError(
                "The 'torchaudio' package is not installed. Re-run autodl_setup.sh "
                "or install UniverSR's dependencies."
            ) from exc

        waveform = audio["waveform"]  # ComfyUI convention: [batch, channels, frames] float32
        sr = int(audio["sample_rate"])
        input_sr = int(input_sr)

        if waveform.ndim != 3:
            raise RuntimeError(
                f"UniverSR Enhance: expected AUDIO waveform with shape [batch, channels, frames], "
                f"got {tuple(waveform.shape)}"
            )
        if sr > OUTPUT_SR:
            raise RuntimeError(
                f"UniverSR Enhance: input sample rate {sr} Hz exceeds {OUTPUT_SR} Hz. "
                f"Resample the audio to <= 48 kHz first (e.g. with a ResampleAudio node)."
            )
        if sr < input_sr:
            raise RuntimeError(
                f"UniverSR Enhance: input sample rate {sr} Hz is below the declared input_sr "
                f"{input_sr} Hz. input_sr declares the source bandwidth (supported: "
                f"{', '.join(INPUT_SRS)} Hz) and must not exceed the file's sample rate."
            )

        net = _get_model(model, _resolve_device(device))
        # Official convention: guidance_scale None (or 0) disables CFG.
        guidance = guidance_scale if guidance_scale > 0 else None

        outs = []
        with tempfile.TemporaryDirectory(prefix="universr-") as tmpdir:
            # UniverSR's official API takes a wav file path (model loads/resamples
            # internally), so stage each batch item through a temp file.
            for i, item in enumerate(waveform):  # item: [channels, frames]
                path = os.path.join(tmpdir, f"input_{i}.wav")
                torchaudio.save(path, item.float().cpu(), sr)
                out = net.enhance(
                    path,
                    input_sr=input_sr,
                    ode_method=ode_method,
                    ode_steps=ode_steps,
                    guidance_scale=guidance,
                )
                out = out.detach().float().cpu()
                if out.ndim == 1:
                    out = out.unsqueeze(0)  # [frames] -> [1, frames] (mono)
                elif out.ndim == 3 and out.shape[0] == 1:
                    out = out.squeeze(0)  # [1, channels, frames] -> [channels, frames]
                elif out.ndim != 2:
                    raise RuntimeError(
                        f"UniverSR Enhance: unexpected output shape {tuple(out.shape)} "
                        f"from UniverSR.enhance()"
                    )
                outs.append(out)

        try:
            batch = torch.stack(outs)  # -> [batch, channels, frames]
        except RuntimeError as exc:
            raise RuntimeError(
                f"UniverSR Enhance: cannot stack per-batch outputs (unequal shapes?): {exc}"
            ) from exc

        return ({"waveform": batch, "sample_rate": OUTPUT_SR},)


NODE_CLASS_MAPPINGS = {"UniverSREnhance": UniverSREnhance}
NODE_DISPLAY_NAME_MAPPINGS = {"UniverSREnhance": "UniverSR Enhance"}

__all__ = ["NODE_CLASS_MAPPINGS", "NODE_DISPLAY_NAME_MAPPINGS"]


if __name__ == "__main__":
    # Minimal self-check: node wiring + clear-error path (no GPU/package needed;
    # the sr < input_sr check fires before the model is touched).
    node = UniverSREnhance()
    assert node.RETURN_TYPES == ("AUDIO",)
    assert node.INPUT_TYPES()["required"]["audio"] == ("AUDIO",)
    fake = {"waveform": torch.zeros(1, 2, 100), "sample_rate": 8000}
    try:
        node.enhance(fake, MODEL_CHOICES[0], "16000", "midpoint", 4, 1.5)
    except RuntimeError as e:
        assert "below the declared input_sr" in str(e), e
    else:
        raise AssertionError("expected a clear error when sr < input_sr")
    assert _resolve_device("auto") in ("cuda", "cpu")
    assert _resolve_device("cuda:1") == "cuda:1"
    print("universr node self-check ok")
'''
with open(os.path.join(universr_dir, "__init__.py"), "w", encoding="utf-8") as _f:
    _f.write(_COMFY_UNIVERSR_SRC)
print(f"[comfyui] comfy-universr node written to {universr_dir}")

print(f"[comfyui] ComfyUI + comfy-mss + Manager + comfy-universr + output-swap installed at {COMFYUI_DIR}")

print("\nInstallation complete. Run the next cell to start the server.")

## 2. Mount Google Drive (persistence)

Everything that should survive a disconnect lives under `DRIVE_ROOT` (default `/content/drive/MyDrive/pymss`):

| Path | Holds |
|---|---|
| `models/` | ComfyUI checkpoints, VAEs, LoRAs, diffusion models (`--models-directory`) |
| `output/` | separated audio — download it from Google Drive (`--output-directory`) |
| `input/` | **staging inbox** for audio you want to feed in — Step 3 copies it to local `input/` |
| `user/` | ComfyUI settings + Manager config, incl. `security_level` (`--user-directory`) |
| `hf/` | HuggingFace / ModelScope weight cache — comfy-mss and UniverSR weights download here |

`HF_HOME`, `HUGGINGFACE_HUB_CACHE` and `MODELSCOPE_CACHE` are redirected into `hf/`, which is the same convention `start_comfyui.sh` uses on a persistent VM. Model weights fetched at runtime land there, so they are not re-downloaded after a reconnect.

### Uploading audio without the slow path

ComfyUI's own `input/` is deliberately **not** on Drive — it is written synchronously on upload and Drive's FUSE mount buffers the whole file and flushes it on close, so pointing `--input-directory` at Drive makes every upload feel stalled. Step 3 therefore leaves `input/` and `temp/` on local disk.

For a fast upload, stage the file instead of pushing it through the tunnel:

1. Open Google Drive in a browser and drop the audio into `pymss/input/`.
2. Re-run Step 3 (the sync happens before ComfyUI starts, because `Load Audio` snapshots its file list at node-definition time).
3. Pick the file in the `Load Audio` node — it now sits in the local `input/`.

To sync without restarting, run `sync_inbox()` in a new cell — but the `Load Audio` dropdown only re-lists files when ComfyUI restarts.

This is usually much faster than uploading through the Cloudflare tunnel, which is also capped per request by Cloudflare (roughly 100 MB on the free tier) and competes with the tunnel's own latency.

> A Colab runtime is wiped when it disconnects, **but Drive survives**. Leave `ENABLE_DRIVE_PERSIST` on unless you are only testing: with it off, every restart re-downloads the models and every output is lost.

> Drive storage counts against your Google quota, and writes through the Drive FUSE mount are slow. Separation output written straight into `output/` is noticeably slower than ephemeral disk; if that matters, turn persistence off for throwaway runs.

In [ ]:
#@title 2. Mount Google Drive (persistence) { display-mode: "form" }
# ============================================================
# Google Drive storage for models / output / config / caches
# ============================================================
# Colab wipes /content on disconnect. Everything this notebook needs to survive
# is therefore kept on Drive and handed to ComfyUI through its own path flags
# (Step 3). Set ENABLE_DRIVE_PERSIST = False to run fully ephemeral.

ENABLE_DRIVE_PERSIST = True #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/pymss" #@param {type:"string"}

import os, shutil

if ENABLE_DRIVE_PERSIST:
    from google.colab import drive
    drive.mount('/content/drive')

    # One subdir per kind of state, so Drive stays browsable and nothing collides.
    # ComfyUI's input/ is deliberately NOT persisted: uploads are written
    # synchronously, and Drive's FUSE mount buffers the whole file and flushes it
    # on close(), which turns every audio upload into a multi-second stall.
    # Step 3 keeps input/ + temp/ on local disk instead.
    PERSIST = {
        "models": os.path.join(DRIVE_ROOT, "models"),   # checkpoints/loras/vae/... (ComfyUI layout)
        "output": os.path.join(DRIVE_ROOT, "output"),   # generated tracks
        "user":   os.path.join(DRIVE_ROOT, "user"),     # ComfyUI settings + Manager config.ini
        "hf":     os.path.join(DRIVE_ROOT, "hf"),       # HuggingFace / ModelScope weight cache
    }
    for _kind, _path in PERSIST.items():
        os.makedirs(_path, exist_ok=True)

    # Staging inbox: drop audio here from the Drive web UI and Step 3 copies it
    # into the local input/ on start. Big files upload to Drive far faster than
    # through the tunnel, and Drive->VM reads are cheap.
    DRIVE_INBOX = os.path.join(DRIVE_ROOT, "input")
    os.makedirs(DRIVE_INBOX, exist_ok=True)

    # Model weights fetched at runtime by comfy-mss / comfy-universr go through
    # HF_HOME, not through ComfyUI's models/ -- redirect both caches or they are
    # re-downloaded on every session. (Same convention as autodl_setup.sh.)
    os.environ["HF_HOME"] = PERSIST["hf"]
    os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(PERSIST["hf"], "hub")
    os.environ["MODELSCOPE_CACHE"] = os.path.join(PERSIST["hf"], "modelscope")

    print(f'[drive] Google Drive mounted, persistent root: {DRIVE_ROOT}')
    for _kind, _path in PERSIST.items():
        try:
            free = f"  ({shutil.disk_usage(_path).free / 1024**3:.0f} GB free)"
        except OSError:
            free = ""
        print(f'[drive]   {_kind:6s} -> {_path}{free}')
    print(f'[drive]   inbox  -> {DRIVE_INBOX}  (drop audio here; Step 3 syncs it to local input/)')
    print(f'[drive] HF_HOME={os.environ["HF_HOME"]}')
else:
    # Ephemeral mode: later cells fall back to ComfyUI's own default paths.
    PERSIST = {}
    DRIVE_INBOX = None
    print('[drive] Persistence disabled -- models, output and config are lost on disconnect')

## 3. Start server


In [ ]:
#@title 3. Start server + tunnel { display-mode: "form" }
# ============================================================
# pymss Colab - Step 3: Start / Stop server + tunnel
# ============================================================
# Click ▶ to start, click ■ to stop everything.

import subprocess, time, urllib.request, sys, threading, os, re, shutil


# ---- Persistence (Step 2): drive-backed paths, else ComfyUI defaults ----
PERSIST = globals().get("PERSIST", {})
COMFYUI_DIR = globals().get("COMFYUI_DIR", "/content/ComfyUI")

def _p(kind):
    """Drive path for `kind`, or None to let ComfyUI use its built-in default."""
    return PERSIST.get(kind)

# Drive staging inbox (Step 2). Big audio uploads to Drive far faster than through
# the tunnel, so the recommended path is: drop files there, restart Step 3, and
# they land in ComfyUI's local input/.
DRIVE_INBOX = globals().get("DRIVE_INBOX")

def sync_inbox():
    """Copy audio staged in the Drive inbox into ComfyUI's *local* input dir.

    Runs before ComfyUI starts because LoadAudio snapshots its file list when the
    node is defined -- files copied later need a restart to show up.
    """
    if not DRIVE_INBOX or not os.path.isdir(DRIVE_INBOX):
        return 0
    dst_dir = os.path.join(COMFYUI_DIR, "input")
    os.makedirs(dst_dir, exist_ok=True)
    copied = 0
    for name in sorted(os.listdir(DRIVE_INBOX)):
        src = os.path.join(DRIVE_INBOX, name)
        if not os.path.isfile(src):
            continue
        dst = os.path.join(dst_dir, name)
        try:
            if os.path.isfile(dst) and os.path.getsize(dst) == os.path.getsize(src):
                continue
            shutil.copyfile(src, dst)
            copied += 1
        except OSError as e:
            print(f"[input] skip {name}: {e}")
    if copied:
        print(f"[input] synced {copied} file(s) from Drive inbox -> {dst_dir}")
    return copied

# ---- Select mode ----
MODE = "webui" #@param ["webui", "comfyui"]

PORT = 8188 #@param {type:"integer"}

SERVER_LOG = "/tmp/pymss_server.log"
TUNNEL_LOG = "/tmp/cloudflared.log"

URL_RE = re.compile(r"(https://[a-z0-9\-]+\.trycloudflare\.com)")

def stream_log(proc, log_path):
    """Read log file and print new lines to cell output."""
    last_pos = 0
    while proc.poll() is None:
        try:
            with open(log_path, "r") as f:
                f.seek(last_pos)
                new = f.read()
                if new:
                    print(new, end="", flush=True)
                    last_pos = f.tell()
        except FileNotFoundError:
            pass
        time.sleep(0.5)

def wait_tunnel_url(log_path, timeout=30):
    """Poll a cloudflared log until its https://*.trycloudflare.com URL shows up."""
    start = time.time()
    while time.time() - start < timeout:
        try:
            with open(log_path, "r") as f:
                m = URL_RE.search(f.read())
            if m:
                return m.group(1)
        except FileNotFoundError:
            pass
        time.sleep(0.5)
    return None

def cleanup_all():
    """Terminate all processes and clean up."""
    global server_proc, tunnel_proc, server_log_f, tunnel_log_f
    for name, proc in (("tunnel", tunnel_proc), ("server", server_proc)):
        if proc and proc.poll() is None:
            print(f"\n[{name}] Stopping...")
            proc.terminate()
            try:
                proc.wait(timeout=5)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
            print(f"[{name}] Stopped.")
    # Close log file handles
    for f in [server_log_f, tunnel_log_f]:
        if f and not f.closed:
            f.close()
    # Clean up temp log files
    for p in [TUNNEL_LOG]:
        if os.path.exists(p):
            try:
                os.unlink(p)
            except OSError:
                pass

server_proc = None
tunnel_proc = None
server_log_f = None
tunnel_log_f = None

try:
    # ---- Start server ----
    if MODE == "comfyui":
        print("[comfyui] Starting ComfyUI server...")
        sync_inbox()
        # Relocate ComfyUI's state through its own flags rather than patching
        # folder_paths: these are the supported switches for models/output/user.
        # input/ and temp/ are left on local disk on purpose -- ComfyUI streams
        # uploads straight to disk and Drive's FUSE mount flushes the whole file
        # on close(), so persisting input/ turns every audio upload into a stall.
        comfy_cmd = [sys.executable, "main.py",
                     "--listen", "0.0.0.0", "--port", str(PORT),
                     "--enable-cors-header", "--fast", "--enable-manager"]
        for flag, kind in (("--models-directory", "models"),
                           ("--output-directory", "output"),
                           ("--user-directory", "user")):
            path = _p(kind)
            if path:
                os.makedirs(path, exist_ok=True)
                comfy_cmd += [flag, path]
        # Manager reads security_level from the *user directory*, which may now
        # live on Drive; write its config there or it defaults to `normal` and
        # rejects --listen 0.0.0.0. (Step 1 skips this -- the path is unknown there.)
        mgr_dir = os.path.join(_p("user") or os.path.join(COMFYUI_DIR, "user"), "__manager")
        os.makedirs(mgr_dir, exist_ok=True)
        mgr_config = os.path.join(mgr_dir, "config.ini")
        if not os.path.isfile(mgr_config):
            with open(mgr_config, "w") as f:
                f.write("[default]\nsecurity_level = weak\n")
            print(f"[manager] Created {mgr_config} with security_level = weak")

        print(f"[comfyui] Persistence: {sorted(PERSIST) if PERSIST else 'OFF (ephemeral)'}")
        server_log_f = open(SERVER_LOG, "w")
        server_proc = subprocess.Popen(
            comfy_cmd,
            cwd=COMFYUI_DIR,
            stdout=server_log_f, stderr=subprocess.STDOUT,
        )
        server_log_thread = threading.Thread(target=stream_log, args=(server_proc, SERVER_LOG), daemon=True)
        server_log_thread.start()
        for _ in range(180):
            try:
                urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=2)
                print(f"[comfyui] Server ready on port {PORT}.")
                break
            except Exception:
                time.sleep(1)
        else:
            raise RuntimeError("ComfyUI failed to start within 180s.")
    else:
        print("[pymss] Starting WebUI server (model loads on first request)...")
        server_log_f = open(SERVER_LOG, "w")
        server_proc = subprocess.Popen(
            ["pymss", "serve", "--debug",
             "--host", "0.0.0.0", "--port", str(PORT),
             "--device", "auto", "--source", "modelscope", "--webui"],
            stdout=server_log_f, stderr=subprocess.STDOUT,
        )
        server_log_thread = threading.Thread(target=stream_log, args=(server_proc, SERVER_LOG), daemon=True)
        server_log_thread.start()
        for _ in range(120):
            try:
                urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2)
                print(f"[pymss] Server ready on port {PORT}.")
                break
            except Exception:
                time.sleep(1)
        else:
            raise RuntimeError("pymss server failed to start within 120s.")

    # ---- Start tunnel ----
    print("[tunnel] Starting cloudflared...")
    tunnel_log_f = open(TUNNEL_LOG, "w")
    tunnel_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--protocol", "http2", "--url", f"http://127.0.0.1:{PORT}"],
        stdout=tunnel_log_f, stderr=subprocess.STDOUT,
    )
    tunnel_log_f.close()  # Close handle so we can re-open for reading
    public_url = wait_tunnel_url(TUNNEL_LOG, timeout=30)
    if not public_url:
        raise RuntimeError("Tunnel failed to start within 30s.")

    # Print URL info
    print(f"\n{'='*60}")
    print(f"  Mode:        {MODE}")
    print(f"  Public URL:  {public_url}")
    if MODE == "comfyui":
        print(f"  WebUI:       {public_url}/")
        out = _p("output")
        if out:
            print(f"  输出目录:    {out}  (Google Drive，可在 Drive 网页下载)")
        print(f"  上传音频:    上传先落到本地 /content/ComfyUI/input/（不占 Drive，不卡）")
    else:
        print(f"  WebUI:       {public_url}/ui/")
        print(f"  API docs:    {public_url}/docs")
    print(f"{'='*60}\n")
    print("Click ■ to stop.")

    # Block until user clicks stop button
    server_proc.wait()
    print("[server] Process exited.")

except KeyboardInterrupt:
    cleanup_all()

except Exception as e:
    cleanup_all()
    raise

---

### Quick Reference

**Mode switch:** Change `MODE = "webui"` to `MODE = "comfyui"` in Step 3, then re-run Step 3.

**WebUI (pymss):** Open URL → browse catalog → load model → upload audio → separate.

**ComfyUI (pymss):** Open URL → build/load workflow with MSS Separate nodes → queue prompt. An `UniverSR Enhance` node is also installed (audio super-resolution to 48 kHz, not separation); it chains as `Load Audio → UniverSR Enhance → MSS Separate`.

**Output swap:** drag one node's output onto another node's output slot of the same type to re-home every downstream link to the dragged source. When the dragged node has one free matching input it is also spliced into the chain (hold Alt to skip). Toggle under Settings → Touch Tools → Output Swap; requires frontend ≥ 1.43, otherwise it quietly no-ops.

**Persistence:** Step 2 mounts Drive and points models, output, config and the HF cache at `DRIVE_ROOT` (default `/content/drive/MyDrive/pymss`). Setting `ENABLE_DRIVE_PERSIST = False` falls back to ephemeral paths. To reclaim space, delete the `models/` or `output/` subfolder from Drive.

**Getting audio in (fast):** upload the file to the Drive `input/` inbox from a browser, then re-run Step 3 — it is copied into ComfyUI's local `input/` before startup. Pushing audio through the ComfyUI upload dialog works too but is slow (Drive FUSE is not involved; the tunnel + Cloudflare's per-request cap are).

**Getting your results:** separated tracks are written to the Drive `output/` folder (printed by Step 3). Open Google Drive in a browser and download them from there.

**Python API:**
```python
from pymss import MSSeparator
with MSSeparator.from_model_name("bs_roformer_voc_hyperacev2", download=True, device="auto", output_format="wav", store_dirs="results") as sep:
    sep.process_folder("/path/to/audio")
```

**Cleanup:**
```python
cleanup_all()  # or click ■
```